# 06 - Validation and reconciliation

Cross-checks before publishing INSIGHTS. Every claim downstream rests on these passing.

Checks:
1. Row-count and total-GMV reconciliation against the raw CSVs.
2. Python monthly city GMV from notebook 03 = SQL B1 result, rupee-exact, every (month, city) cell.
3. Order-status counts sum to total orders; shipments tie 1-to-1 to non-cancelled orders.
4. B2 cohort spot-check: 5 random customers per cohort, trace their delivered-order history by hand.
5. Edge cases: customers with zero delivered orders, Lost shipments, InTransit handling.

In [1]:
import pandas as pd
import numpy as np
import duckdb
from pathlib import Path

RAW = Path('..').resolve().parent
PROC = Path('../data/processed').resolve()

con = duckdb.connect()
for name in ['customers', 'sellers', 'products', 'orders', 'order_items', 'shipments']:
    con.execute(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_csv_auto('{RAW / (name + '.csv')}')")

orders = pd.read_parquet(PROC / 'orders_enriched.parquet')
items = pd.read_parquet(PROC / 'items_enriched.parquet')
ships = pd.read_parquet(PROC / 'shipments_enriched.parquet')
b1 = pd.read_parquet(PROC / 'b1_monthly_city_metrics.parquet')
print('parquet rows  - orders:', len(orders), ' items:', len(items), ' ships:', len(ships))
print('b1 rows:', len(b1))

parquet rows  - orders: 100000  items: 169929  ships: 91994
b1 rows: 216


## 1. Row counts & GMV vs raw CSVs

In [2]:
raw_counts = con.execute("""
SELECT
    (SELECT COUNT(*) FROM customers)   AS customers,
    (SELECT COUNT(*) FROM sellers)     AS sellers,
    (SELECT COUNT(*) FROM products)    AS products,
    (SELECT COUNT(*) FROM orders)      AS orders,
    (SELECT COUNT(*) FROM order_items) AS items,
    (SELECT COUNT(*) FROM shipments)   AS shipments
""").df().iloc[0]
print(raw_counts.to_string())

raw_gmv = con.execute("SELECT ROUND(SUM(quantity * unit_price), 2) AS gmv FROM order_items").fetchone()[0]
py_gmv = round(float(items['gmv_line'].sum()), 2)
print(f"\nraw csv gmv:    {raw_gmv:,.2f}")
print(f"parquet gmv:    {py_gmv:,.2f}")
assert abs(raw_gmv - py_gmv) < 0.01, 'GMV mismatch between CSV and parquet'
assert len(orders) == int(raw_counts['orders']), 'orders row mismatch'
assert len(items) == int(raw_counts['items']), 'items row mismatch'
assert len(ships) == int(raw_counts['shipments']), 'shipments row mismatch'
print('\nrow counts and total gmv reconcile.')

customers     25000
sellers         400
products       3000
orders       100000
items        169929
shipments     91994

raw csv gmv:    3,391,994,349.00
parquet gmv:    3,391,994,349.00

row counts and total gmv reconcile.


## 2. Monthly city GMV: parquet vs SQL B1

Recompute the (month, city) GMV in pandas from the enriched orders/items, run the B1 query in DuckDB on the raw CSVs, join the two and assert the rupee-exact match across all 216 (month, city) cells.

In [3]:
py_city = (
    orders.assign(month=lambda d: d['created_at'].dt.to_period('M').dt.to_timestamp())
          .groupby(['month', 'city'], as_index=False)['order_gmv'].sum()
          .rename(columns={'order_gmv': 'gmv'})
          .assign(gmv=lambda d: d['gmv'].round(2))
)

sql_city = con.execute("""
WITH order_gmv AS (
    SELECT order_id, SUM(quantity * unit_price) AS gmv
    FROM order_items GROUP BY order_id
)
SELECT DATE_TRUNC('month', o.created_at)::DATE AS month,
       c.city,
       ROUND(SUM(og.gmv), 2) AS gmv
FROM orders o
JOIN customers c USING (customer_id)
LEFT JOIN order_gmv og USING (order_id)
GROUP BY 1, 2
""").df()
sql_city['month'] = pd.to_datetime(sql_city['month'])

merged = py_city.merge(sql_city, on=['month', 'city'], suffixes=('_py', '_sql'))
merged['diff'] = (merged['gmv_py'] - merged['gmv_sql']).abs()
print('rows:', len(merged), ' max abs diff:', merged['diff'].max())
assert merged['diff'].max() < 0.01, 'monthly city GMV diverges between python and SQL'
assert len(merged) == len(py_city) == len(sql_city), 'row count differs'
print('python and SQL B1 agree on all (month, city) GMVs.')

rows: 216  max abs diff: 0.0
python and SQL B1 agree on all (month, city) GMVs.


## 3. Status counts and shipment coverage

In [4]:
status = orders['status'].value_counts()
print(status.to_string())
assert int(status.sum()) == len(orders) == 100_000

non_cancelled = (orders['status'] != 'Cancelled').sum()
print(f"\nnon-cancelled orders:  {non_cancelled}")
print(f"shipments:             {len(ships)}")
assert non_cancelled == len(ships), 'shipment count should equal non-cancelled orders'

cancelled_with_ship = orders[orders['status'] == 'Cancelled']['order_id'].isin(ships['order_id']).sum()
assert cancelled_with_ship == 0, 'cancelled orders should not have shipments'
print('\nstatus counts sum to 100k; cancelled orders carry no shipment.')

status
Delivered    82069
Cancelled     8006
Shipped       5043
Returned      4882

non-cancelled orders:  91994
shipments:             91994

status counts sum to 100k; cancelled orders carry no shipment.


## 4. B2 cohort spot-check

Pick 5 random customers from each first-order cohort (OnTime / Delayed) and walk their delivered-order history to confirm the 90-day repeat flag was set correctly.

In [5]:
first = pd.read_parquet(PROC / 'customer_first_order.parquet')
delivered = orders[orders['delivered_at'].notna()].copy()
cutoff = delivered['delivered_at'].max()

eligible = first[first['first_delivered_at'] + pd.Timedelta(days=90) <= cutoff].copy()

rng = np.random.default_rng(7)
samples = pd.concat([
    eligible[eligible['first_order_delay_status'] == 'OnTime'].sample(5, random_state=int(rng.integers(1e9))),
    eligible[eligible['first_order_delay_status'] == 'Delayed'].sample(5, random_state=int(rng.integers(1e9))),
])

rows = []
for _, r in samples.iterrows():
    cid = r['customer_id']
    hist = delivered[delivered['customer_id'] == cid].sort_values('delivered_at')
    first_at = r['first_delivered_at']
    repeat = ((hist['delivered_at'] > first_at) & (hist['delivered_at'] <= first_at + pd.Timedelta(days=90))).any()
    rows.append({
        'customer_id': cid,
        'cohort': r['first_order_delay_status'],
        'first_delivered_at': first_at.date(),
        'delivered_orders_total': len(hist),
        'repeated_within_90d': bool(repeat),
    })
pd.DataFrame(rows)

,customer_id,cohort,first_delivered_at,delivered_orders_total,repeated_within_90d
0,C08537,OnTime,2025-03-25,2,True
1,C03195,OnTime,2024-11-20,1,False
2,C09994,OnTime,2024-10-29,4,False
3,C13992,OnTime,2024-08-05,4,True
4,C01156,OnTime,2024-08-18,4,True
5,C04257,Delayed,2025-01-12,4,False
6,C00791,Delayed,2024-07-22,8,True
7,C11759,Delayed,2025-06-11,2,False
8,C13507,Delayed,2025-08-25,3,True
9,C19676,Delayed,2024-08-14,6,True


## 5. Edge cases

In [6]:
delivered_per_customer = delivered.groupby('customer_id')['order_id'].nunique()
all_customers = orders['customer_id'].unique()
zero_delivered = len(set(all_customers) - set(delivered_per_customer.index))
print(f"customers with zero delivered orders:  {zero_delivered}")

intransit = (ships['delivery_status'] == 'InTransit').sum()
lost = (ships['delivery_status'] == 'Lost').sum()
print(f"InTransit shipments (excluded from delayed denom):  {intransit}")
print(f"Lost shipments (counted as delayed):                {lost}")

evaluable = (ships['delivery_status'] != 'InTransit').sum()
delayed = ((ships['delivery_status'] != 'OnTime') & (ships['delivery_status'] != 'InTransit')).sum()
rate = delayed / evaluable
print(f"\nevaluable shipments:  {evaluable}")
print(f"delayed:              {delayed}  ({rate:.2%})")
assert abs(rate - 0.2676) < 0.002, 'headline delayed rate drifted from INSIGHTS'
print('\ndelayed-rate matches INSIGHTS headline (~26.8%).')

customers with zero delivered orders:  497
InTransit shipments (excluded from delayed denom):  5043
Lost shipments (counted as delayed):                430

evaluable shipments:  86951
delayed:              23269  (26.76%)

delayed-rate matches INSIGHTS headline (~26.8%).


All checks pass. Numbers in INSIGHTS.md, the SQL results, and the parquet outputs are mutually consistent and tied back to the raw CSVs.